# Quick SDFC Demo

This notebook is a short CPU demonstration of the SDFC benchmark and replay path. It is intentionally small and should not be read as a reproduction of the final multi-seed results.

For the final claim, use the curated CSVs in `results/main_tables/` and the reproduction scripts in `docs/README_REPRODUCIBILITY.md`.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
ROOT

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import torch

from src.data.sdfc_dataset import build_data_bundle
from src.data.sdfc_generator import DEFAULT_BENCHMARK_SEED, generate_all_splits
from src.models.film import FiLMClassifier
from src.training.train import train_sequential
from src.utils.seed import set_seed

device = torch.device("cpu")
set_seed(7)
torch.set_num_threads(1)
device

## Build a tiny SDFC run

The final experiments use 10,000 training examples per task and 5 seeds. This demo uses a much smaller split so it runs quickly on CPU.

In [ ]:
NUM_TASKS = 4
N_TRAIN = 384
N_VAL = 128
N_TEST = 256
EPOCHS = 2
BATCH_SIZE = 64

splits = generate_all_splits(
    ROOT,
    num_tasks=NUM_TASKS,
    n_train=N_TRAIN,
    n_val=N_VAL,
    n_test=N_TEST,
    benchmark_seed=DEFAULT_BENCHMARK_SEED,
)
data_bundle = build_data_bundle(splits, NUM_TASKS)
data_bundle.input_dim, len(data_bundle.train)

In [ ]:
def run_demo(replay_fraction: float):
    replay_size = int(round(N_TRAIN * replay_fraction))
    set_seed(7)
    model = FiLMClassifier(
        input_dim=data_bundle.input_dim,
        hidden_dims=[32],
        scenario="shared_head",
        num_tasks=NUM_TASKS,
        mode="full",
    )
    result = train_sequential(
        model=model,
        data_bundle=data_bundle,
        scenario="shared_head",
        num_tasks=NUM_TASKS,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        lr=1e-3,
        apical_lr_mult=1.0,
        weight_decay=0.0,
        device=device,
        run_name=f"quick_demo_replay_{replay_fraction:g}",
        benchmark="sdfc",
        seed=7,
        replay_size_per_task=replay_size,
    )
    return {
        "replay_fraction": replay_fraction,
        "replay_examples_per_task": replay_size,
        "average_accuracy_%": 100.0 * result.summary["average_accuracy"],
        "average_forgetting_%": 100.0 * result.summary["average_forgetting"],
    }

demo_rows = [run_demo(0.0), run_demo(0.02)]
demo_df = pd.DataFrame(demo_rows)
demo_df

## Demo result

The exact values below are intentionally demonstrative. Small CPU runs are noisy. The purpose is to exercise the benchmark generation, model path, sequential training loop, and replay buffer.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))
axes[0].bar(demo_df["replay_examples_per_task"].astype(str), demo_df["average_accuracy_%"], color="#2563eb")
axes[0].set_title("Demo accuracy")
axes[0].set_ylabel("Accuracy (%)")
axes[1].bar(demo_df["replay_examples_per_task"].astype(str), demo_df["average_forgetting_%"], color="#f97316")
axes[1].set_title("Demo forgetting")
axes[1].set_ylabel("Forgetting (%)")
for ax in axes:
    ax.set_xlabel("Replay examples per task")
    ax.grid(axis="y", alpha=0.25)
    ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
plt.show()

## Final curated result

For the actual repository claim, use the final multi-seed table.

In [ ]:
final_table = pd.read_csv(ROOT / "results" / "main_tables" / "table_main_performance.csv")
final_table